#### import 套件

In [1]:
import numpy as np
import os
import pathlib, joblib
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from pathlib import Path

from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

2025-07-30 15:37:06.702548: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-30 15:37:06.730950: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-30 15:37:07.137165: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


#### load power trace

In [2]:
trace_num = 500
order = '1'
opt = 'o3' # o0 or o3
algo = 'PQClean' # orgPoly or PQClean or pqm4
platform = 'CW308_STM32F4' # CW308_STM32F3 or CW308_STM32F4 or CWLITEARM
BATCH_SIZE = 512

trace_path = f'traces_attack/{trace_num}_3/traces_unfixed_message_{trace_num}_picoscope_{platform}_{opt}_{algo}_poi_{order}'


t_raw = np.load(f'{trace_path}/trace.npy')
l_raw = np.load(f'{trace_path}/label.npy')

#### 確認trace和label大小

In [3]:
print("Size of Trace: ", end="")
print(t_raw.shape)
print(t_raw)
print("\nSize of Message: ", end="")
print(l_raw.shape)
print(l_raw)

Size of Trace: (16000, 4808)
[[ 2.78964717e-03  4.12342815e-03  4.37675497e-03 ... -1.23763887e-02
  -1.08533757e-02 -6.91917959e-03]
 [-9.52264681e-04 -1.14454890e-03 -1.20864363e-03 ... -4.31571237e-03
  -1.65120254e-03 -7.10841167e-03]
 [-8.27127335e-04 -9.52264681e-04 -6.34843120e-04 ... -2.05622024e-02
  -1.71987547e-02 -1.50408985e-02]
 ...
 [ 6.10426077e-05 -3.17421560e-04 -5.09705775e-04 ... -2.69686241e-02
  -2.84916372e-02 -1.91643267e-02]
 [-8.27127335e-04 -2.56378953e-04 -1.92284214e-04 ... -9.58368942e-03
  -1.28189476e-02 -1.24374313e-02]
 [-2.03271884e-03 -1.84043462e-03 -1.20559150e-03 ... -1.30722744e-02
  -1.37071176e-02 -1.06610914e-02]]

Size of Message: (16000,)
[120. 202. 251. ...  57.  16. 152.]


#### 對trace進行正歸化

In [4]:
scaler_path = f'model/{platform}/2000/best_model/scaler.pkl'
scaler = joblib.load(scaler_path)

t = scaler.transform(t_raw)
print('Scaled trace shape:', t.shape)

Scaled trace shape: (16000, 4808)


#### 確認trace和label大小

In [5]:
print("Size of Trace: ", end="")
print(t.shape)
print(t)
print("\nSize of Message: ", end="")
print(l_raw.shape)
print(l_raw[:50])

Size of Trace: (16000, 4808)
[[ 2.29101068  2.81404707  2.88905029 ... -0.48640869 -0.20271691
   0.18726661]
 [ 0.20493774 -0.01393837 -0.11682723 ...  0.61756504  1.08873793
   0.1621179 ]
 [ 0.27470037  0.08928474  0.1919733  ... -1.60752097 -1.09324248
  -0.89209976]
 ...
 [ 0.76984493  0.43008484  0.2593181  ... -2.48493136 -2.67811243
  -1.44009824]
 [ 0.27470037  0.46285408  0.43014392 ... -0.10392631 -0.47856995
  -0.5461022 ]
 [-0.39740306 -0.38750771 -0.11518468 ... -0.58171577 -0.60321783
  -0.31002883]]

Size of Message: (16000,)
[120. 202. 251.  36.  25. 144.  34. 122. 191.   9. 139. 239. 219. 176.
 181. 204.  24.  44. 166. 163. 237.  86. 210.  28. 195. 203.  71.  17.
 216. 165. 158.  10. 229. 201.  71. 224. 202.  69.  51. 177. 112. 214.
 106. 171.  74. 188. 116. 209.  58.  38.]


#### 載入模型

In [6]:
model_path = f'model/{platform}/2000/best_model/best_model.keras'
model  = tf.keras.models.load_model(model_path)
print(f"Loaded model from {model_path}")

2025-07-30 15:37:09.866632: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-07-30 15:37:09.890867: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-07-30 15:37:09.890977: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Loaded model from model/CW308_STM32F4/2000/best_model/best_model.keras


In [7]:
preds_raw = model.predict(t, verbose=1)
print(preds_raw.shape)

500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 572us/step 
(16000, 256)


In [8]:
print(preds_raw)
print(np.max(preds_raw[3]))
print(np.argmax(preds_raw[3]))

[[3.50039941e-12 7.23416878e-13 3.96553043e-14 ... 2.11793805e-09
  3.04469384e-12 1.10118857e-13]
 [8.66432148e-10 1.69062355e-12 4.25565645e-07 ... 1.10551657e-14
  2.40401102e-12 2.76151376e-14]
 [2.48625718e-18 7.53408384e-16 4.28384346e-16 ... 1.24433754e-07
  7.07039627e-09 7.74117416e-06]
 ...
 [1.23459271e-10 4.93283835e-07 9.27216068e-14 ... 5.03864624e-11
  1.38272160e-16 1.94667314e-15]
 [4.14537033e-03 5.02199100e-07 6.37495532e-06 ... 5.70990317e-19
  7.46343806e-20 1.78069812e-21]
 [1.10521469e-07 1.89343929e-10 1.78496895e-09 ... 2.78169458e-12
  1.05465278e-12 1.16265555e-14]]
0.99773896
36


In [9]:
preds = preds_raw.reshape(trace_num, 32, -1)
labels = l_raw.reshape(trace_num, 32)
print(preds.shape)
print(preds)
print(labels.shape)
print(labels)

(500, 32, 256)
[[[3.50039941e-12 7.23416878e-13 3.96553043e-14 ... 2.11793805e-09
   3.04469384e-12 1.10118857e-13]
  [8.66432148e-10 1.69062355e-12 4.25565645e-07 ... 1.10551657e-14
   2.40401102e-12 2.76151376e-14]
  [2.48625718e-18 7.53408384e-16 4.28384346e-16 ... 1.24433754e-07
   7.07039627e-09 7.74117416e-06]
  ...
  [1.26841910e-13 8.50569948e-10 1.69685157e-15 ... 2.55339149e-14
   4.28953687e-19 2.88886211e-17]
  [2.76602040e-11 2.35595607e-14 4.09962952e-09 ... 1.41276439e-14
   2.30274966e-11 1.79036253e-14]
  [3.29577489e-07 7.01704250e-10 1.07707537e-03 ... 6.15382936e-17
   1.38638475e-16 7.41132984e-18]]

 [[2.69712146e-14 7.70645353e-11 3.99563679e-16 ... 1.52641590e-07
   1.51043186e-13 5.90912388e-12]
  [1.53041967e-13 1.10540888e-09 9.09469016e-16 ... 1.33502653e-09
   5.05188767e-16 5.83137813e-13]
  [1.09932097e-09 1.16036779e-06 5.72815395e-07 ... 7.83148706e-15
   1.07821111e-15 1.02404081e-13]
  ...
  [9.07115449e-09 1.25992365e-05 9.40621019e-12 ... 2.97395639

In [10]:
top1_idx = np.argmax(preds, axis=2)
# print(top1_idx.shape)
# print(top1_idx)

local_correct = np.sum(top1_idx == labels)
correct_traces = np.sum(np.all(top1_idx == labels, axis=1))
print(f"Total correct bytes: {local_correct} / {trace_num*32}")
print(f"Total fully correct traces: {correct_traces} / {trace_num}")

top2 = np.argsort(preds, axis=2)[:, :, -2:]    # (trace, byte, 2)
guess1 = top2[:, :, 1]                         # (trace, byte)
local_errors  = np.sum(guess1 != labels)
global_errors = np.sum(np.any(guess1 != labels, axis=1))
byte_acc  = (trace_num*32 - local_errors) / (trace_num*32)
trace_acc = (trace_num   - global_errors) / trace_num
print(f"Byte-Level Accuracy : {byte_acc*100:.4f}%")
print(f"Trace-Level Accuracy: {trace_acc*100:.4f}%")

Total correct bytes: 15885 / 16000
Total fully correct traces: 398 / 500
Byte-Level Accuracy : 99.2812%
Trace-Level Accuracy: 79.6000%
